# Manual Backpropagation & Training Self-Attention in Pure NumPy

Building training from scratch in **pure NumPy** (calculating the backward pass and weight gradients analytically) is the ultimate test of understanding!

In this notebook, we will:
1. **Forward Pass:** Compute $Q, K, V$, Attention Matrix $A$, Contextual Output $H$, and Loss $L$ in NumPy.
2. **Backward Pass:** Calculate exact analytical gradients $\frac{\partial L}{\partial W_Q}, \frac{\partial L}{\partial W_K}, \frac{\partial L}{\partial W_V}$ using matrix calculus.
3. **Gradient Descent Update:** Manually update $W_Q, W_K, W_V$ using $W \leftarrow W - \eta \cdot \nabla_W L$.
4. **Watch it Learn:** Observe loss decreasing and attention weights specializing over training epochs!

## Step 1: Dataset & Helper Functions

In [ ]:
import numpy as np

np.random.seed(42)

# Mini Vocabulary
vocab = {"river": 0, "bank": 1, "overflowed": 2, "money": 3, "account": 4}
vocab_size = len(vocab)
emb_dim = 8
head_dim = 4

# Create fixed embedding table E (vocab_size, emb_dim)
E = np.random.randn(vocab_size, emb_dim) * 0.1

# Dataset:
# Sentence 1: "river bank overflowed" -> Label 0 (Nature)
# Sentence 2: "money bank account"   -> Label 1 (Finance)
dataset = [
    ([vocab["river"], vocab["bank"], vocab["overflowed"]], 0),
    ([vocab["money"], vocab["bank"], vocab["account"]], 1)
]

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

print("Setup complete! Embeddings initialized.")

## Step 2: Initialize Trainable Parameters ($W_Q, W_K, W_V, W_c$)

In [ ]:
# Projection Weights (emb_dim, head_dim) -> (8, 4)
W_Q = np.random.randn(emb_dim, head_dim) * 0.1
W_K = np.random.randn(emb_dim, head_dim) * 0.1
W_V = np.random.randn(emb_dim, head_dim) * 0.1

# Classifier Weight (head_dim, num_classes) -> (4, 2)
W_c = np.random.randn(head_dim, 2) * 0.1
b_c = np.zeros((1, 2))

print("Trainable parameter weights initialized.")

## Step 3: Forward Pass, Backward Pass & Gradient Math in NumPy

Let's write explicit functions for Forward and Backward passes!

### Gradient Formula Derivations:
1. $\delta_{\text{logits}} = \text{probs} - y_{\text{onehot}}$ (Cross-Entropy derivative)
2. $\frac{\partial L}{\partial H_{\text{bank}}} = \delta_{\text{logits}} W_c^T$
3. $\frac{\partial L}{\partial V} = A^T \cdot \frac{\partial L}{\partial H}$
4. $\frac{\partial L}{\partial A} = \frac{\partial L}{\partial H} \cdot V^T$
5. $\delta_S = \frac{1}{\sqrt{d_k}} A \odot \left( \delta_A - \sum (\delta_A \odot A) \right)$ (Softmax derivative)
6. $\frac{\partial L}{\partial Q} = \delta_S \cdot K \quad \text{and} \quad \frac{\partial L}{\partial K} = \delta_S^T \cdot Q$
7. $\frac{\partial L}{\partial W_Q} = X^T \cdot \frac{\partial L}{\partial Q}, \quad \frac{\partial L}{\partial W_K} = X^T \cdot \frac{\partial L}{\partial K}, \quad \frac{\partial L}{\partial W_V} = X^T \cdot \frac{\partial L}{\partial V}$

In [ ]:
def forward_pass(token_ids, W_Q, W_K, W_V, W_c, b_c):
    # 1. Lookup Input Embeddings X (seq_len, emb_dim)
    X = E[token_ids]
    
    # 2. Linear projections Q, K, V
    Q = X @ W_Q  # (seq_len, head_dim)
    K = X @ W_K  # (seq_len, head_dim)
    V = X @ W_V  # (seq_len, head_dim)
    
    # 3. Raw Scores S and Scaled Scores
    d_k = head_dim
    S = (Q @ K.T) / np.sqrt(d_k)  # (seq_len, seq_len)
    
    # 4. Softmax Attention Weights A
    A = softmax(S, axis=-1)  # (seq_len, seq_len)
    
    # 5. Contextual Outputs H
    H = A @ V  # (seq_len, head_dim)
    
    # 6. Select 'bank' vector (index 1)
    h_bank = H[1:2, :]  # (1, head_dim)
    
    # 7. Classifier logits & softmax probabilities
    logits = h_bank @ W_c + b_c  # (1, 2)
    probs = softmax(logits, axis=-1)
    
    cache = (X, Q, K, V, S, A, H, h_bank, logits, probs)
    return probs, cache

def backward_pass(probs, label, cache, W_Q, W_K, W_V, W_c):
    X, Q, K, V, S, A, H, h_bank, logits, probs = cache
    seq_len = X.shape[0]
    d_k = head_dim
    
    # One-hot target
    y_onehot = np.zeros((1, 2))
    y_onehot[0, label] = 1.0
    
    # 1. dL/d_logits
    d_logits = probs - y_onehot  # (1, 2)
    
    # 2. Classifier gradients
    dW_c = h_bank.T @ d_logits   # (head_dim, 2)
    db_c = d_logits              # (1, 2)
    
    # 3. dL/dH (gradient backpropagating into Contextual Matrix H)
    dH = np.zeros_like(H)        # (seq_len, head_dim)
    dH[1:2, :] = d_logits @ W_c.T
    
    # 4. dL/dV and dL/dA
    dV = A.T @ dH                # (seq_len, head_dim)
    dA = dH @ V.T                # (seq_len, seq_len)
    
    # 5. dL/dS (Softmax backward)
    sum_dA_A = np.sum(dA * A, axis=-1, keepdims=True)
    dS = (A * (dA - sum_dA_A)) / np.sqrt(d_k)  # (seq_len, seq_len)
    
    # 6. dL/dQ and dL/dK
    dQ = dS @ K                  # (seq_len, head_dim)
    dK = dS.T @ Q                # (seq_len, head_dim)
    
    # 7. Final Projection Weight Gradients dL/dW_Q, dL/dW_K, dL/dW_V
    dW_Q = X.T @ dQ              # (emb_dim, head_dim)
    dW_K = X.T @ dK              # (emb_dim, head_dim)
    dW_V = X.T @ dV              # (emb_dim, head_dim)
    
    return dW_Q, dW_K, dW_V, dW_c, db_c

## Step 4: Pure NumPy Training Loop (SGD)

In [ ]:
learning_rate = 0.1
epochs = 200

print("Training Self-Attention with Pure NumPy Backpropagation...\n")

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for token_ids, label in dataset:
        # 1. Forward Pass
        probs, cache = forward_pass(token_ids, W_Q, W_K, W_V, W_c, b_c)
        
        # 2. Compute Loss (-log probability of correct class)
        loss = -np.log(probs[0, label] + 1e-12)
        total_loss += loss
        
        # 3. Backward Pass (Analytical Gradients)
        dW_Q, dW_K, dW_V, dW_c, db_c = backward_pass(probs, label, cache, W_Q, W_K, W_V, W_c)
        
        # 4. SGD Weight Updates: W = W - lr * dW
        W_Q -= learning_rate * dW_Q
        W_K -= learning_rate * dW_K
        W_V -= learning_rate * dW_V
        W_c -= learning_rate * dW_c
        b_c -= learning_rate * db_c
        
    if epoch % 40 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Total Loss: {total_loss:.4f}")

## Step 5: Test Trained NumPy Self-Attention & Visualize Attention Weights!

In [ ]:
print("\n================ RESULTS AFTER NUMPY TRAINING ================\n")

# Test Sentence 1: "river bank overflowed"
probs1, cache1 = forward_pass(dataset[0][0], W_Q, W_K, W_V, W_c, b_c)
attn1 = cache1[5]
pred1 = np.argmax(probs1)

print("Sentence 1: ['river', 'bank', 'overflowed']")
print(f"Predicted Class: {pred1} (0 = Nature/Water)")
print("Attention Weights for 'bank' (Row 1 -> ['river', 'bank', 'overflowed']):")
print(attn1[1].round(4))
print("-" * 55)

# Test Sentence 2: "money bank account"
probs2, cache2 = forward_pass(dataset[1][0], W_Q, W_K, W_V, W_c, b_c)
attn2 = cache2[5]
pred2 = np.argmax(probs2)

print("Sentence 2: ['money', 'bank', 'account']")
print(f"Predicted Class: {pred2} (1 = Finance)")
print("Attention Weights for 'bank' (Row 1 -> ['money', 'bank', 'account']):")
print(attn2[1].round(4))